# TP — Support de code : l'intelligence artificielle connexionniste

Notebook compagnon de la séance *« Comprendre le fonctionnement des réseaux de neurones »* (INSA Strasbourg).
Il reprend les démonstrations PyTorch du support, **dans le même ordre**, pour que vous puissiez les faire tourner vous-même et répondre aux questions des participants.

Conçu pour tourner directement sur **Google Colab**, sans rien installer (PyTorch y est préinstallé). Les données des avocats sont recopiées directement dans ce notebook : pas besoin du fichier `data/avocats.csv`.

Les parties « **Exercice** » vont un peu plus loin que le support — elles sont volontairement **moins commentées**.

## Sommaire

1. Un neurone artificiel : le problème des avocats
2. La limite d'un seul neurone : le problème XOR
3. Une couche cachée résout XOR : le perceptron multicouche (MLP)
4. Bonus — activations, softmax, pertes (annexes du support)

## 0. Préparatifs

In [3]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from matplotlib.colors import ListedColormap
from io import StringIO

plt.rcParams.update({"figure.figsize": (8, 4.5), "font.size": 12,
                      "axes.spines.top": False, "axes.spines.right": False})

BLEU, ORANGE, VERT, ROUGE = "#0072B2", "#E69F00", "#009E73", "#D55E00"
CMAP = ListedColormap([ORANGE, BLEU])          # classe 0 = orange, classe 1 = bleu
FOND = ListedColormap(["#FCF1DE", "#E3EFF7"])  # mêmes teintes, en fond

OSError: [WinError 1114] Произошел сбой в программе инициализации библиотеки динамической компоновки (DLL). Error loading "D:\lessons\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

Deux fonctions utilitaires, réutilisées pour toutes les démonstrations :
- `frontiere()` : dessine la frontière de décision apprise par un modèle PyTorch à 2 entrées
- `exactitude()` : proportion de prédictions correctes (seuil à 0,5)

In [ ]:
def frontiere(modele, X_np, y_np, titre, xlabel, ylabel, marge=0.6):
    # évalue le modèle sur une grille fine du plan (x1, x2) pour colorer les deux régions
    g1 = np.linspace(X_np[:, 0].min() - marge, X_np[:, 0].max() + marge, 300)
    g2 = np.linspace(X_np[:, 1].min() - marge, X_np[:, 1].max() + marge, 300)
    G1, G2 = np.meshgrid(g1, g2)
    grille = torch.tensor(np.c_[G1.ravel(), G2.ravel()], dtype=torch.float32)
    with torch.no_grad():                                    # pas besoin de gradients pour juste évaluer
        Z = torch.sigmoid(modele(grille)).numpy().reshape(G1.shape)

    plt.contourf(G1, G2, Z, levels=[0, .5, 1], cmap=FOND)
    plt.contour(G1, G2, Z, levels=[.5], colors="#3A3A3A", linewidths=1.8)  # la frontière elle-même
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap=CMAP, edgecolor="white", s=58, linewidth=.9)
    plt.title(titre)
    plt.xlabel(xlabel); plt.ylabel(ylabel)


def exactitude(modele, X, y):
    # part des observations bien classées : on seuille la probabilité prédite à 0,5
    with torch.no_grad():
        return (((torch.sigmoid(modele(X)) > .5).float() == y).float().mean().item())

## 1. Un neurone artificiel : le problème des avocats

Trier des avocats en *bon à manger* / *pas encore*, à partir de deux mesures : la teinte
(0 = vert clair, 10 = presque noir) et la fermeté (0 = souple, 10 = dur). 80 fruits, cible binaire.

In [ ]:
# les données du support (data/avocats.csv), recopiées ici pour que le notebook soit autonome
csv_avocats = """teinte,fermete,bon
5.25,7.79,0
6.26,3.54,1
4.28,7.36,0
3.87,6.21,0
4.46,7.24,0
8.01,5.24,1
6.52,3.18,1
3.64,5.82,0
3.97,6.97,0
5.61,2.85,1
4.82,6.02,0
0.46,5.69,0
7.46,4.87,1
7.13,3.83,1
4.59,7.55,0
6.33,3.31,1
4.17,7.79,0
5.76,2.26,1
2.3,6.78,0
7.25,5.21,1
7.55,4.92,1
3.23,6.47,0
7.01,1.97,1
4.96,1.21,1
2.36,4.86,0
3.03,7.11,0
3.57,5.31,0
7.15,3.44,1
7.91,3.31,1
5.76,3.83,1
1.31,7.07,0
4.76,5.84,0
6.54,2.42,1
7.21,2.21,1
5.83,3.64,1
6.35,4.66,1
4.39,8.09,0
3.11,8.69,0
4.63,3.7,1
2.82,5.16,0
7.28,0.72,1
7.71,4.37,1
3.16,4.96,0
6.63,4.01,1
8.29,5.27,1
3.74,6.81,0
5.38,6.75,0
1.97,6.93,0
2.27,6.25,0
6.77,4.79,1
2.13,8.24,0
1.68,6.12,0
7.49,3.24,1
2.34,6.82,0
6.13,4.66,1
8.19,3.65,1
2.59,5.94,0
3.86,3.04,1
2.27,9.31,0
1.61,6.59,0
5.56,3.13,1
6.38,3.07,1
4.67,6.98,0
4.86,6.78,0
6.89,3.52,1
8.05,2.6,1
6.35,3.92,1
0.85,4.61,0
3.65,6.76,0
6.91,1.52,1
4.54,6.79,0
7.03,2.58,1
7.97,4.75,1
5.37,4.06,1
3.94,5.57,0
2.54,3.78,0
5.07,1.45,1
2.74,7.06,0
7.83,3.02,1
3.07,7.56,0
"""
donnees = pd.read_csv(StringIO(csv_avocats))
donnees.head(4)

Du tableau aux tenseurs PyTorch : `X` (80×2) et `y` (80×1).

In [ ]:
X = torch.tensor(donnees[["teinte", "fermete"]].values, dtype=torch.float32)
y = torch.tensor(donnees["bon"].values, dtype=torch.float32).reshape(-1, 1)
print(X.shape, y.shape)

Un neurone en PyTorch tient en une ligne : `nn.Linear(2, 1)` — 2 entrées, 1 sortie.

In [ ]:
torch.manual_seed(63)
neurone = nn.Linear(2, 1)  # un neurone = une combinaison pondérée des 2 entrées, plus un biais

print("poids initiaux :", neurone.weight.data.numpy().round(3))
print("biais initial  :", neurone.bias.data.numpy().round(3))

Avant tout apprentissage, les poids sont aléatoires : la frontière l'est aussi.

In [ ]:
frontiere(neurone, X.numpy(), y.numpy().ravel(),
          "Frontière AVANT apprentissage (poids aléatoires)",
          "teinte (0 = vert clair → 10 = presque noir)", "fermeté (0 = souple → 10 = dur)")
plt.show()
print(f"exactitude : {exactitude(neurone, X, y):.0%}")

### La boucle d'apprentissage : les 4 étapes qui reviennent toujours

`BCELoss` (*binary cross-entropy*) mesure l'écart entre la probabilité prédite et la cible binaire ;
`SGD` met à jour les poids dans le sens indiqué par les gradients, avec un pas `lr` (taux d'apprentissage).

In [ ]:
perte = nn.BCELoss()                                          # la fonction de coût à minimiser
optimiseur = torch.optim.SGD(neurone.parameters(), lr=0.05)   # lr : amplitude de chaque mise à jour

for etape in range(400):
    optimiseur.zero_grad()                     # 0. remettre les gradients précédents à zéro
    proba = torch.sigmoid(neurone(X))          # 1. propagation avant : X -> z -> a (probabilité entre 0 et 1)
    erreur = perte(proba, y)                   # 2. mesurer l'erreur L(ŷ, y)
    erreur.backward()                          # 3. rétropropagation : calcule les gradients ∂L/∂w, ∂L/∂b
    optimiseur.step()                          # 4. corrige les poids dans le sens qui fait baisser l'erreur

Après 400 étapes, les mêmes poids ont déplacé la frontière.

In [ ]:
frontiere(neurone, X.numpy(), y.numpy().ravel(),
          "Frontière APRÈS apprentissage",
          "teinte (0 = vert clair → 10 = presque noir)", "fermeté (0 = souple → 10 = dur)")
plt.show()
print("poids appris :", neurone.weight.data.numpy().round(2), " biais :", neurone.bias.data.numpy().round(2))
print(f"exactitude d'entraînement : {exactitude(neurone, X, y):.0%}")

### Exercice 1 — aller plus loin

Refaites l'entraînement avec un taux d'apprentissage `lr` très petit (`0.001`) puis très grand (`2.0`).
Que se passe-t-il sur la courbe de la frontière et sur l'exactitude finale ?

In [ ]:
for lr_test in [0.001, 2.0]:
    torch.manual_seed(63)
    n_test = nn.Linear(2, 1)
    opt_test = torch.optim.SGD(n_test.parameters(), lr=lr_test)
    for _ in range(400):
        opt_test.zero_grad()
        erreur = nn.BCELoss()(torch.sigmoid(n_test(X)), y)
        erreur.backward()
        opt_test.step()
    print(f"lr={lr_test:<6} -> exactitude finale : {exactitude(n_test, X, y):.0%}, erreur finale : {erreur.item():.3f}")

# À vous : appelez frontiere(n_test, ...) pour visualiser l'effet sur la frontière elle-même.

## 2. La limite d'un seul neurone : le problème XOR

| x₁ | x₂ | ET | OU | **XOR** |
|:--:|:--:|:--:|:--:|:--:|
| 0 | 0 | 0 | 0 | **0** |
| 0 | 1 | 0 | 1 | **1** |
| 1 | 0 | 0 | 1 | **1** |
| 1 | 1 | 1 | 1 | **0** |

Un neurone ne peut tracer qu'une droite (un hyperplan). *ET* et *OU* sont séparables par une droite ; **XOR ne l'est pas**.

In [ ]:
torch.manual_seed(6)
base = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
X_xor = base.repeat(30, 1) + 0.08 * torch.randn(120, 2)      # 120 points bruités (30 par cas)
y_xor = torch.tensor([0., 1., 1., 0.]).repeat(30).reshape(-1, 1)

On emballe la boucle d'entraînement précédente dans une fonction, pour la réutiliser telle quelle.

In [ ]:
def entrainer(modele, X, y, iterations=3000, lr=0.1):
    # exactement les 4 étapes vues plus haut, répétées, avec l'historique de l'erreur en plus
    perte, optimiseur = nn.BCELoss(), torch.optim.SGD(modele.parameters(), lr=lr)
    historique = []
    for _ in range(iterations):
        optimiseur.zero_grad()
        erreur = perte(torch.sigmoid(modele(X)), y)
        erreur.backward()
        optimiseur.step()
        historique.append(erreur.item())
    return historique

In [ ]:
torch.manual_seed(6)
perceptron = nn.Linear(2, 1)                              # un seul neurone, comme pour les avocats
hist_perceptron = entrainer(perceptron, X_xor, y_xor)      # entrainer() : la même boucle, réutilisée

print(f"erreur finale : {hist_perceptron[-1]:.3f}   (0,693 = ln 2 : perte d'une prédiction proche de 0,5 partout)")
print(f"exactitude d'entraînement    : {exactitude(perceptron, X_xor, y_xor):.0%}")

In [ ]:
frontiere(perceptron, X_xor.numpy(), y_xor.numpy().ravel(),
          "Un seul neurone : la frontière ne peut pas séparer XOR",
          "$x_1$", "$x_2$", marge=0.35)
plt.show()

> Ce n'est pas un problème d'entraînement trop court ou mal réglé : c'est une **impossibilité structurelle**.

### Exercice 2

Recommencez avec 2 ou 3 autres valeurs pour `torch.manual_seed(...)`. Le neurone finit-il par
résoudre XOR avec un tirage de poids initiaux différent ?

In [ ]:
for graine in [0, 1, 42]:
    torch.manual_seed(graine)
    p = nn.Linear(2, 1)
    h = entrainer(p, X_xor, y_xor)
    print(f"seed={graine:<3} -> exactitude finale : {exactitude(p, X_xor, y_xor):.0%}")

## 3. Une couche cachée résout XOR : le perceptron multicouche (MLP)

On remplace le neurone unique par **deux neurones cachés** suivis d'une activation non linéaire
(`tanh`), puis une couche de sortie. Le reste — la boucle `entrainer()` — ne change pas.

In [ ]:
torch.manual_seed(6)
reseau = nn.Sequential(
    nn.Linear(2, 2),      # couche cachée : 2 neurones
    nn.Tanh(),            # activation non linéaire : sans elle, empiler des Linear resterait linéaire
    nn.Linear(2, 1),      # couche de sortie
)
hist_reseau = entrainer(reseau, X_xor, y_xor)        # la MÊME fonction d'entraînement que pour le neurone seul

print(f"erreur finale : {hist_reseau[-1]:.3f}   (contre {hist_perceptron[-1]:.3f} sans couche cachée)")
print(f"exactitude d'entraînement    : {exactitude(reseau, X_xor, y_xor):.0%}")

In [ ]:
frontiere(reseau, X_xor.numpy(), y_xor.numpy().ravel(),
          "Avec une couche cachée : les deux diagonales sont séparées",
          "$x_1$", "$x_2$", marge=0.35)
plt.show()

Les deux courbes d'apprentissage, superposées : le réseau à couche cachée s'échappe du niveau du hasard, pas le neurone seul.

In [ ]:
plt.plot(hist_perceptron, color=ORANGE, lw=2, label="1 neurone (aucune couche cachée)")
plt.plot(hist_reseau, color=BLEU, lw=2, label="2 neurones cachés + tanh")
plt.axhline(0.693, color="#999999", ls=":", lw=1.4)
plt.text(1500, 0.72, "0,693 = niveau du hasard", color="#777777", fontsize=11)
plt.xlabel("itérations"); plt.ylabel("erreur (entropie croisée)")
plt.legend()
plt.show()

### Exercice 3 — aller plus loin

Remplacez `nn.Tanh()` par `nn.ReLU()` dans le réseau. L'apprentissage réussit-il toujours ?
Essayez aussi plus de neurones cachés (`nn.Linear(2, 8)` puis `nn.Linear(8, 1)`).

In [ ]:
torch.manual_seed(6)
reseau_relu = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))
hist_relu = entrainer(reseau_relu, X_xor, y_xor)
print(f"ReLU, 2 neurones cachés   -> exactitude : {exactitude(reseau_relu, X_xor, y_xor):.0%}")

torch.manual_seed(6)
reseau_8 = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
hist_8 = entrainer(reseau_8, X_xor, y_xor)
print(f"tanh, 8 neurones cachés   -> exactitude : {exactitude(reseau_8, X_xor, y_xor):.0%}")

## 4. Bonus — activations, softmax, pertes (annexes du support)

### Les fonctions d'activation, tracées

Sigmoïde (sortie binaire), tanh (couches cachées), ReLU (le choix par défaut aujourd'hui).

In [ ]:
z = torch.linspace(-4, 4, 200)

plt.plot(z, torch.sigmoid(z), color=BLEU, label="sigmoïde")
plt.plot(z, torch.tanh(z), color=ORANGE, label="tanh")
plt.plot(z, torch.relu(z), color=VERT, label="ReLU")
plt.axhline(0, color="#DDDDDD", lw=0.8)
plt.xlabel("z"); plt.ylabel("activation(z)")
plt.legend()
plt.show()

### Softmax : des scores bruts vers une distribution de probabilité (annexe An5)

Trois scores bruts (chat / chien / cheval) ne somment à rien de particulier ; softmax les transforme
en probabilités positives qui somment exactement à 1.

In [ ]:
scores = torch.tensor([5.0, 4.0, 2.0])
probas = torch.softmax(scores, dim=0)  # softmax : exponentielle de chaque score, divisée par la somme

for etiquette, p in zip(["chat", "chien", "cheval"], probas):
    print(f"{etiquette:<7} : {p.item():.2f}")
print("somme   :", round(probas.sum().item(), 3))

### Trois tâches, trois pertes (annexe An7)

Le choix de la fonction de perte dépend uniquement de la nature de `y` — exactement comme en séance 5.

In [ ]:
# régression : un nombre réel quelconque -> MSELoss
y_num, y_num_hat = torch.tensor([3.2]), torch.tensor([2.9])
print("MSE  (régression)            :", nn.MSELoss()(y_num_hat, y_num).item())

# classification binaire : 1 neurone + sigmoïde -> BCELoss
y_bin, p_bin = torch.tensor([1.0]), torch.tensor([0.8])
print("BCE  (classification binaire):", nn.BCELoss()(p_bin, y_bin).item())

# classification multi-classe : K neurones (logits, sans softmax manuel) -> CrossEntropyLoss
logits = torch.tensor([[2.0, 1.0, 0.1]])
cible = torch.tensor([0])  # la bonne classe est la première
print("CE   (multi-classe)          :", nn.CrossEntropyLoss()(logits, cible).item())

### Exercice 4

`nn.CrossEntropyLoss` applique déjà softmax en interne (cf. support) : si on lui passe des
probabilités déjà « softmaxées » au lieu des scores bruts (*logits*), le résultat est faux.
Vérifiez-le : comparez `CrossEntropyLoss()(logits, cible)` à `CrossEntropyLoss()(torch.softmax(logits, dim=1), cible)`.

In [ ]:
print("avec les logits bruts        :", nn.CrossEntropyLoss()(logits, cible).item())
print("avec softmax appliqué en trop :", nn.CrossEntropyLoss()(torch.softmax(logits, dim=1), cible).item())

## Pour continuer

- **PyTorch** : `pytorch.org/docs` (référence), `pytorch.org/tutorials` (prise en main guidée)
- **TensorFlow Playground** (`playground.tensorflow.org`) : un réseau dans le navigateur, ajoutez des couches, changez l'activation, regardez la frontière bouger
- **3Blue1Brown**, série *Neural Networks* : la meilleure explication visuelle de la rétropropagation
- Le schéma reste le même qu'en séance 5 : `modele(X)` puis apprentissage par minimisation d'une erreur — seuls le modèle et l'algorithme d'optimisation changent.